# Reinforcement learning from Bellman operators to entropy regularization

This notebook is an executable mathematical refresher, not an API survey.  We
begin with exact finite-state operators, introduce sampling one approximation
at a time, and end with readable PyTorch implementations of DQN, REINFORCE,
advantage actor--critic, DDPG, and SAC.

**Execution modes.** `QUICK=True` gives a CPU-friendly structural smoke run.
Set it to `False` for plots with lower Monte Carlo error. Deep-control results
are intentionally diagnostics, not benchmark claims.


In [ ]:
from __future__ import annotations

from collections import deque
from dataclasses import dataclass
import math
import random

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

SEED = 7
QUICK = True
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
available_styles = set(plt.style.available)
plot_style = next(
    (
        style
        for style in ("seaborn-v0_8-whitegrid", "seaborn-whitegrid", "ggplot")
        if style in available_styles
    ),
    "default",
)
plt.style.use(plot_style)


## 1. Finite Markov decision processes

A discounted MDP is a tuple

$$\mathcal M=(\mathcal S,\mathcal A,P,R,\gamma),$$

with transition kernel $P(s'\mid s,a)$, conditional expected reward
$R(s,a,s')=\mathbb E[R_{t+1}\mid S_t=s,A_t=a,S_{t+1}=s']$, and
$\gamma\in[0,1)$.  A (stationary Markov) policy $\pi(a\mid s)$ induces

$$G_t=\sum_{k=0}^{\infty}\gamma^kR_{t+k+1},\qquad
  V^\pi(s)=\mathbb E_\pi[G_t\mid S_t=s],\qquad
  Q^\pi(s,a)=\mathbb E_\pi[G_t\mid S_t=s,A_t=a].$$

The Bellman expectation operator is

$$({\cal T}^{\pi}V)(s)=\sum_a\pi(a\mid s)\sum_{s'}P(s'\mid s,a)
   [R(s,a,s')+\gamma V(s')],$$

while $({\cal T}^*V)(s)=\max_a\sum_{s'}P(s'\mid s,a)
[R(s,a,s')+\gamma V(s')]$. Equivalently, the action-value equations are

$$Q^\pi(s,a)=\sum_{s'}P(s'\mid s,a)\left[R(s,a,s')+
  \gamma\sum_{a'}\pi(a'\mid s')Q^\pi(s',a')\right],$$
$$Q^*(s,a)=\sum_{s'}P(s'\mid s,a)\left[R(s,a,s')+
  \gamma\max_{a'}Q^*(s',a')\right].$$

Both value operators are $\gamma$-contractions in the sup norm
for discounted finite MDPs. Episodic problems instead obtain finiteness from
absorption; continuing problems need discounting or an average-reward
formulation.


In [ ]:
# Four states, two actions; state 3 is absorbing. The risky branch via state 2
# can pay more but sometimes falls back to state 1.
nS, nA, gamma = 4, 2, 0.92
P = np.zeros((nS, nA, nS))
R = np.zeros_like(P)
P[0, 0, 1] = 1.0
P[0, 1, 2], P[0, 1, 1] = 0.75, 0.25
P[1, 0, 3], P[1, 1, 1] = 1.0, 1.0
P[2, 0, 3], P[2, 1, 0] = 1.0, 1.0
P[3, :, 3] = 1.0
R[1, 0, 3] = 1.0
R[2, 0, 3] = 2.2
R[1, 1, 1] = -0.08
terminal = np.array([False, False, False, True])
assert np.allclose(P.sum(axis=-1), 1.0)


## 2. Dynamic programming: exact expectations

Policy evaluation repeatedly applies ${\cal T}^{\pi}$. Policy iteration
alternates exact/approximate evaluation with greedy improvement. Value
iteration applies ${\cal T}^*$ directly. The terminal mask below enforces
zero continuation after absorption, making the episodic convention explicit.


In [ ]:
def q_from_v(P, R, V, gamma, terminal):
    continuation = gamma * V * (~terminal)
    return np.sum(P * (R + continuation[None, None, :]), axis=2)


def policy_evaluation(P, R, policy, gamma, terminal, tol=1e-12):
    V = np.zeros(P.shape[0])
    residuals = []
    while True:
        Q = q_from_v(P, R, V, gamma, terminal)
        updated = np.sum(policy * Q, axis=1)
        updated[terminal] = 0.0
        residuals.append(np.max(np.abs(updated - V)))
        V = updated
        if residuals[-1] < tol:
            return V, np.asarray(residuals)


def policy_iteration(P, R, gamma, terminal):
    policy = np.full(P.shape[:2], 1 / P.shape[1])
    changes = []
    while True:
        V, _ = policy_evaluation(P, R, policy, gamma, terminal)
        greedy = np.argmax(q_from_v(P, R, V, gamma, terminal), axis=1)
        improved = np.eye(P.shape[1])[greedy]
        improved[terminal] = 1 / P.shape[1]
        changes.append(np.mean(np.argmax(policy, axis=1) != greedy))
        if np.array_equal(np.argmax(policy, axis=1)[~terminal], greedy[~terminal]):
            return V, policy, np.asarray(changes)
        policy = improved


def value_iteration(P, R, gamma, terminal, tol=1e-12):
    V = np.zeros(P.shape[0])
    residuals = []
    while True:
        Q = q_from_v(P, R, V, gamma, terminal)
        updated = Q.max(axis=1)
        updated[terminal] = 0.0
        residuals.append(np.max(np.abs(updated - V)))
        V = updated
        if residuals[-1] < tol:
            Q = q_from_v(P, R, V, gamma, terminal)
            return V, Q, np.argmax(Q, axis=1), np.asarray(residuals)


uniform = np.full((nS, nA), 0.5)
V_uniform, eval_residuals = policy_evaluation(P, R, uniform, gamma, terminal)
V_pi, pi, policy_changes = policy_iteration(P, R, gamma, terminal)
V_star, Q_star, greedy, vi_residuals = value_iteration(P, R, gamma, terminal)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].bar(np.arange(nS) - .18, V_uniform, .36, label=r"$V^{uniform}$")
ax[0].bar(np.arange(nS) + .18, V_star, .36, label=r"$V^*$")
ax[0].set(xlabel="state", ylabel="value", title=f"Greedy actions: {greedy}")
ax[0].legend()
ax[1].semilogy(eval_residuals, label="policy evaluation")
ax[1].semilogy(vi_residuals, label="value iteration")
ax[1].set(xlabel="sweep", ylabel=r"$\|V_{k+1}-V_k\|_\infty$",
          title="Bellman residual")
ax[1].legend()
plt.tight_layout()


## 3. Monte Carlo: replace expectations by complete returns

First-visit prediction averages $G_t$ only at the first occurrence of a state
in each episode. Exploring-starts control (used here for clarity) estimates
$Q$ and greedifies after every return:

$$Q(S_t,A_t)\leftarrow Q(S_t,A_t)+\frac{1}{N(S_t,A_t)}
  [G_t-Q(S_t,A_t)].$$

Ordinary on-policy $\epsilon$-soft control is usually easier to deploy than
exploring starts. Off-policy Monte Carlo requires importance ratios, whose
heavy tails can dominate finite-sample behavior.


In [ ]:
def sample_transition(s, a, rng):
    next_s = rng.choice(nS, p=P[s, a])
    return next_s, R[s, a, next_s], bool(terminal[next_s])


def generate_episode(policy, rng, start=(0, None), max_steps=100):
    s, forced_action = start
    episode = []
    for t in range(max_steps):
        a = forced_action if t == 0 and forced_action is not None else rng.choice(nA, p=policy[s])
        next_s, reward, done = sample_transition(s, a, rng)
        episode.append((s, a, reward))
        s = next_s
        if done:
            break
    return episode


def first_visit_mc_prediction(policy, episodes, rng):
    values, counts = np.zeros(nS), np.zeros(nS)
    history = []
    for _ in range(episodes):
        trajectory = generate_episode(policy, rng)
        G, returns = 0.0, []
        for s, _, reward in reversed(trajectory):
            G = reward + gamma * G
            returns.append((s, G))
        visited = set()
        for s, G in reversed(returns):
            if s not in visited:
                visited.add(s)
                counts[s] += 1
                values[s] += (G - values[s]) / counts[s]
        history.append(values.copy())
    return values, np.asarray(history)


def mc_control(episodes, rng, epsilon=0.8):
    Q, counts = np.zeros((nS, nA)), np.zeros((nS, nA))
    errors = []
    for episode_index in range(episodes):
        s0, a0 = int(rng.integers(0, nS - 1)), int(rng.integers(nA))
        epsilon_t = epsilon / np.sqrt(episode_index + 1)
        greedy_actions = Q.argmax(1)
        policy = np.full((nS, nA), epsilon_t / nA)
        policy[np.arange(nS), greedy_actions] += 1 - epsilon_t
        trajectory = generate_episode(policy, rng, start=(s0, a0))
        returns, G = np.zeros(len(trajectory)), 0.0
        for t in reversed(range(len(trajectory))):
            _, _, reward = trajectory[t]
            G = reward + gamma * G
            returns[t] = G
        visited = set()
        for (s, a, _), G in zip(trajectory, returns, strict=True):
            if (s, a) not in visited:
                visited.add((s, a))
                counts[s, a] += 1
                Q[s, a] += (G - Q[s, a]) / counts[s, a]
        errors.append(np.max(np.abs(Q[~terminal] - Q_star[~terminal])))
    return Q, np.asarray(errors)


episodes = 1_500 if QUICK else 20_000
V_mc, V_mc_path = first_visit_mc_prediction(uniform, episodes, rng)
Q_mc, Q_mc_error = mc_control(episodes, rng)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(np.abs(V_mc_path[:, 0] - V_uniform[0]))
ax[0].set(xlabel="episode", ylabel="absolute error", title="First-visit MC at state 0")
ax[1].plot(Q_mc_error)
ax[1].set(xlabel="episode", ylabel=r"$\|Q-Q^*\|_\infty$", title="MC control")
plt.tight_layout()


## 4. Temporal differences: bootstrap before the episode ends

TD(0) uses

$$\delta_t=R_{t+1}+\gamma V(S_{t+1})-V(S_t),\qquad
  V(S_t)\leftarrow V(S_t)+\alpha\delta_t.$$

The $n$-step target interpolates between one-step TD and Monte Carlo:
$G_{t:t+n}=\sum_{k=0}^{n-1}\gamma^kR_{t+k+1}+\gamma^nV(S_{t+n})$.
For action values, the only difference among several canonical methods is the
bootstrap target:

| method | target after $R_{t+1}$ |
|---|---|
| SARSA | $\gamma Q(S_{t+1},A_{t+1})$ |
| Expected SARSA | $\gamma\sum_a\pi(a\mid S_{t+1})Q(S_{t+1},a)$ |
| Q-learning | $\gamma\max_a Q(S_{t+1},a)$ |
| Double Q | evaluate one estimator's argmax with the other estimator |

SARSA is on-policy and prices exploratory actions into its values. Q-learning
is off-policy. Double Q-learning attacks the maximization bias caused by
selecting and evaluating an action with the same noisy estimator.


In [ ]:
def epsilon_probs(q_row, epsilon):
    probs = np.full(len(q_row), epsilon / len(q_row))
    probs[np.flatnonzero(q_row == q_row.max())] += (1 - epsilon) / np.sum(q_row == q_row.max())
    return probs


def td_prediction(policy, episodes, alpha, n_step=1):
    V = np.zeros(nS)
    errors = []
    local_rng = np.random.default_rng(SEED + n_step)
    for _ in range(episodes):
        trajectory = generate_episode(policy, local_rng)
        states = [x[0] for x in trajectory]
        rewards = [x[2] for x in trajectory]
        for t, s in enumerate(states):
            horizon = min(t + n_step, len(states))
            target = sum(gamma ** k * rewards[t + k] for k in range(horizon - t))
            if horizon < len(states):
                target += gamma ** n_step * V[states[horizon]]
            V[s] += alpha * (target - V[s])
        errors.append(abs(V[0] - V_uniform[0]))
    return V, np.asarray(errors)


def control(algorithm, episodes, alpha=.12, epsilon=.12):
    Q = np.zeros((nS, nA))
    algorithm_seed = {"sarsa": 11, "expected_sarsa": 12, "q_learning": 13}[algorithm]
    local_rng = np.random.default_rng(SEED + algorithm_seed)
    errors, td_errors = [], []
    for _ in range(episodes):
        s = 0
        a = local_rng.choice(nA, p=epsilon_probs(Q[s], epsilon))
        for _ in range(100):
            next_s, reward, done = sample_transition(s, a, local_rng)
            next_probs = epsilon_probs(Q[next_s], epsilon)
            next_a = local_rng.choice(nA, p=next_probs)
            if done:
                target = reward
            elif algorithm == "sarsa":
                target = reward + gamma * Q[next_s, next_a]
            elif algorithm == "expected_sarsa":
                target = reward + gamma * np.dot(next_probs, Q[next_s])
            else:
                target = reward + gamma * Q[next_s].max()
            delta = target - Q[s, a]
            Q[s, a] += alpha * delta
            td_errors.append(delta)
            s, a = next_s, next_a
            if done:
                break
        errors.append(np.max(np.abs(Q[~terminal] - Q_star[~terminal])))
    return Q, np.asarray(errors), np.asarray(td_errors)


def double_q_learning(episodes, alpha=.12, epsilon=.12):
    QA, QB = np.zeros((nS, nA)), np.zeros((nS, nA))
    local_rng = np.random.default_rng(SEED + 91)
    errors, td_errors = [], []
    for _ in range(episodes):
        s = 0
        for _ in range(100):
            a = local_rng.choice(nA, p=epsilon_probs(QA[s] + QB[s], epsilon))
            next_s, reward, done = sample_transition(s, a, local_rng)
            first = bool(local_rng.integers(2))
            update, evaluate = (QA, QB) if first else (QB, QA)
            target = reward if done else reward + gamma * evaluate[next_s, np.argmax(update[next_s])]
            delta = target - update[s, a]
            update[s, a] += alpha * delta
            td_errors.append(delta)
            s = next_s
            if done:
                break
        Q = (QA + QB) / 2
        errors.append(np.max(np.abs(Q[~terminal] - Q_star[~terminal])))
    return (QA + QB) / 2, np.asarray(errors), np.asarray(td_errors)


td_episodes = 1_000 if QUICK else 10_000
_, td0_error = td_prediction(uniform, td_episodes, alpha=.08, n_step=1)
_, td4_error = td_prediction(uniform, td_episodes, alpha=.08, n_step=4)
results = {name: control(name, td_episodes) for name in ("sarsa", "expected_sarsa", "q_learning")}
results["double_q"] = double_q_learning(td_episodes)

fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
ax[0].plot(td0_error, alpha=.8, label="TD(0)")
ax[0].plot(td4_error, alpha=.8, label="4-step TD")
ax[0].set(title="Prediction", xlabel="episode", ylabel=r"$|V(0)-V^\pi(0)|$")
ax[0].legend()
for name, (_, error, _) in results.items():
    ax[1].plot(error, alpha=.8, label=name)
ax[1].set(title="Control", xlabel="episode", ylabel=r"$\|Q-Q^*\|_\infty$")
ax[1].legend(fontsize=8)
for name, (_, _, deltas) in results.items():
    ax[2].hist(deltas[-500:], bins=30, alpha=.35, density=True, label=name)
ax[2].set(title="Late-training TD errors", xlabel=r"$\delta_t$", ylabel="density")
ax[2].legend(fontsize=8)
plt.tight_layout()


### Failure modes and neighboring algorithms

Constant step sizes preserve adaptation but leave a noise floor; Robbins--Monro
schedules converge only under adequate visitation and stationary assumptions.
Bootstrapping, off-policy sampling, and function approximation form the
*deadly triad*. Eligibility traces ($\operatorname{TD}(\lambda)$, SARSA($\lambda$))
mix all $n$-step targets. Expected SARSA removes action-sampling variance;
Q-learning removes behavior-policy bias but can amplify max bias. Inspect TD
error distributions and visits—not only mean return.


## 5. Function approximation

Replace a table with $Q_\theta(s,a)$. Semi-gradient TD minimizes a moving
target, e.g.

$$L(\theta)=\mathbb E[(R+\gamma\max_{a'}Q_{\bar\theta}(S',a')
  -Q_\theta(S,A))^2].$$

A target network $\bar\theta$ slows target drift; replay reduces serial
correlation and reuses data. Neither makes the objective stationary. The code
below is a compact DQN with explicit replay, target synchronization, gradient
clipping, and seeded environments. Install the repository's `notebooks` extra
from a terminal rather than installing PyTorch inside a notebook cell; the
declared platform markers keep PyTorch and NumPy binary-compatible.


In [ ]:
import torch
from torch import nn
from torch.distributions import Categorical, Normal
import torch.nn.functional as F

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class Replay:
    def __init__(self, capacity=100_000):
        self.data = deque(maxlen=capacity)

    def add(self, *transition):
        self.data.append(transition)

    def sample(self, batch_size, rng):
        idx = rng.choice(len(self.data), batch_size, replace=False)
        columns = list(zip(*(self.data[i] for i in idx)))
        return [np.asarray(column) for column in columns]

    def __len__(self):
        return len(self.data)


def mlp(sizes, final=nn.Identity):
    layers = []
    for left, right in zip(sizes[:-2], sizes[1:-1]):
        layers += [nn.Linear(left, right), nn.ReLU()]
    layers += [nn.Linear(sizes[-2], sizes[-1]), final()]
    return nn.Sequential(*layers)


def train_dqn(episodes=40):
    torch.manual_seed(SEED)
    env = gym.make("CartPole-v1")
    env.action_space.seed(SEED)
    online = mlp([4, 64, 64, 2]).to(device)
    target = mlp([4, 64, 64, 2]).to(device)
    target.load_state_dict(online.state_dict())
    optimizer = torch.optim.Adam(online.parameters(), lr=7e-4)
    replay, local_rng = Replay(), np.random.default_rng(SEED)
    returns, losses, steps = [], [], 0
    for episode in range(episodes):
        state, _ = env.reset(seed=SEED + episode)
        total = 0.0
        for _ in range(500):
            epsilon = 0.03 + 0.97 * math.exp(-steps / 1_500)
            if local_rng.random() < epsilon:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    action = int(online(torch.as_tensor(state, dtype=torch.float32, device=device)).argmax())
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            # TimeLimit truncation resets the rollout but is not an absorbing
            # MDP transition, so only true termination masks the target.
            replay.add(state, action, reward, next_state, terminated)
            state, total, steps = next_state, total + reward, steps + 1
            if len(replay) >= 64:
                s, a, r, ns, d = replay.sample(64, local_rng)
                st = torch.as_tensor(s, dtype=torch.float32, device=device)
                at = torch.as_tensor(a, dtype=torch.long, device=device)
                rt = torch.as_tensor(r, dtype=torch.float32, device=device)
                nst = torch.as_tensor(ns, dtype=torch.float32, device=device)
                dt = torch.as_tensor(d, dtype=torch.float32, device=device)
                prediction = online(st).gather(1, at[:, None]).squeeze(1)
                with torch.no_grad():
                    y = rt + .99 * (1 - dt) * target(nst).max(1).values
                loss = F.smooth_l1_loss(prediction, y)
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(online.parameters(), 10.0)
                optimizer.step(); losses.append(float(loss))
            if steps % 250 == 0:
                target.load_state_dict(online.state_dict())
            if done:
                break
        returns.append(total)
    env.close()
    return np.asarray(returns), np.asarray(losses)


dqn_returns, dqn_losses = train_dqn(30 if QUICK else 250)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.3))
ax[0].plot(dqn_returns); ax[0].set(title="DQN: episodic return", xlabel="episode")
ax[1].plot(dqn_losses); ax[1].set(title="DQN: Huber loss", xlabel="gradient step")
plt.tight_layout()


## 6. Policy gradients and actor--critic

The policy-gradient theorem gives

$$\nabla_\theta J(\theta)=\mathbb E_{d^{\pi_\theta},\pi_\theta}
  [\nabla_\theta\log\pi_\theta(A\mid S)Q^{\pi_\theta}(S,A)].$$

REINFORCE substitutes a Monte Carlo return $G_t$ and a baseline $b(S_t)$.
Actor--critic substitutes a bootstrapped critic and uses the TD residual as a
one-step advantage estimate. The implementation below is deliberately a
single-environment advantage actor--critic, rather than claiming the
synchronous parallel actors usually denoted A2C.

The corresponding sample objectives are

$$\Delta\theta_{REINFORCE}\propto
  \sum_t\nabla_\theta\log\pi_\theta(A_t\mid S_t)
  [G_t-b_\phi(S_t)],$$
$$\delta_t=R_{t+1}+\gamma(1-D_{t+1})V_\phi(S_{t+1})-V_\phi(S_t),$$
$$L_V(\phi)=\tfrac12\delta_t^2,\qquad
  L_\pi(\theta)=-\log\pi_\theta(A_t\mid S_t)\,\operatorname{stopgrad}(\delta_t)
  -\beta{\cal H}(\pi_\theta(\cdot\mid S_t)).$$


In [ ]:
class DiscreteActorCritic(nn.Module):
    def __init__(self, obs_dim, n_actions):
        super().__init__()
        self.body = mlp([obs_dim, 64, 64, 64])
        self.policy = nn.Linear(64, n_actions)
        self.value = nn.Linear(64, 1)

    def forward(self, x):
        features = self.body(x)
        return self.policy(features), self.value(features).squeeze(-1)


def train_policy_gradient(method="reinforce", episodes=40):
    torch.manual_seed(SEED + (0 if method == "reinforce" else 1))
    env = gym.make("CartPole-v1")
    env.action_space.seed(SEED)
    model = DiscreteActorCritic(4, 2).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=8e-4)
    episode_returns = []
    diagnostics = {"actor_loss": [], "critic_loss": [], "entropy": [], "gradient_norm": []}
    for episode in range(episodes):
        state, _ = env.reset(seed=SEED + 10_000 + episode)
        logps, values, rewards, entropies, terminations = [], [], [], [], []
        for _ in range(500):
            x = torch.as_tensor(state, dtype=torch.float32, device=device)
            logits, value = model(x)
            distribution = Categorical(logits=logits)
            action = distribution.sample()
            state, reward, terminated, truncated, _ = env.step(int(action))
            logps.append(distribution.log_prob(action)); values.append(value)
            rewards.append(reward); entropies.append(distribution.entropy())
            terminations.append(float(terminated))
            if terminated or truncated:
                break
        raw_returns, G = [], 0.0
        for reward in reversed(rewards):
            G = reward + .99 * G; raw_returns.append(G)
        raw_returns = torch.as_tensor(raw_returns[::-1], dtype=torch.float32, device=device)
        logps, values = torch.stack(logps), torch.stack(values)
        if method == "reinforce":
            # A learned baseline lowers variance; detach keeps this unbiased.
            advantages = raw_returns - values.detach()
            critic_target = raw_returns
        else:
            with torch.no_grad():
                final_state = torch.as_tensor(state, dtype=torch.float32, device=device)
                _, final_value = model(final_state)
                bootstrap = torch.zeros((), device=device) if terminated else final_value
            next_values = torch.cat([values[1:].detach(), bootstrap.reshape(1)])
            rewards_t = torch.as_tensor(rewards, dtype=torch.float32, device=device)
            terminated_t = torch.as_tensor(terminations, dtype=torch.float32, device=device)
            critic_target = rewards_t + .99 * (1 - terminated_t) * next_values
            advantages = critic_target - values.detach()
        normalized_advantages = (advantages - advantages.mean()) / (
            advantages.std(unbiased=False) + 1e-6
        )
        actor_loss = -(logps * normalized_advantages).mean()
        critic_loss = .5 * F.mse_loss(values, critic_target.detach())
        entropy_bonus = torch.stack(entropies).mean()
        loss = actor_loss + critic_loss - .01 * entropy_bonus
        optimizer.zero_grad(); loss.backward()
        gradient_norm = nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        episode_returns.append(sum(rewards))
        diagnostics["actor_loss"].append(float(actor_loss))
        diagnostics["critic_loss"].append(float(critic_loss))
        diagnostics["entropy"].append(float(entropy_bonus))
        diagnostics["gradient_norm"].append(float(gradient_norm))
    env.close()
    return np.asarray(episode_returns), {
        key: np.asarray(values) for key, values in diagnostics.items()
    }


pg_runs = {
    name: train_policy_gradient(name, 30 if QUICK else 250)
    for name in ("reinforce", "advantage_actor_critic")
}
fig, ax = plt.subplots(1, 3, figsize=(14, 3.3))
for name, (returns, diagnostics) in pg_runs.items():
    ax[0].plot(returns, label=name)
    ax[1].plot(diagnostics["actor_loss"], label=name)
    ax[2].plot(diagnostics["critic_loss"], label=f"{name}: critic")
    ax[2].plot(diagnostics["entropy"], linestyle="--", label=f"{name}: entropy")
ax[0].set(title="Policy-gradient returns", xlabel="episode")
ax[1].set(title="Actor loss", xlabel="episode")
ax[2].set(title="Critic loss / policy entropy", xlabel="episode")
for axis in ax: axis.legend()
plt.tight_layout()


## 7. Continuous control: DDPG and SAC

DDPG uses a deterministic actor $\mu_\theta$ and critic $Q_\phi$:

$$\nabla_\theta J\approx\mathbb E[\nabla_a Q_\phi(s,a)|_{a=\mu_\theta(s)}
  \nabla_\theta\mu_\theta(s)],$$
$$y=r+\gamma Q_{\bar\phi}(s',\mu_{\bar\theta}(s')).$$

Exploration is external action noise, so its scale and temporal structure are
consequential. SAC instead optimizes return plus entropy,

$$J(\pi)=\mathbb E\sum_t\gamma^t[R_{t+1}+\alpha
  {\cal H}(\pi(\cdot\mid S_t))],$$

with a squashed Gaussian actor and clipped double critics. SAC's stochastic
policy makes exploration part of the objective. Its sampled soft target and
actor objective are

$$y=r+\gamma(1-d)\left[\min_{i=1,2}Q_{\bar\phi_i}(s',a')
  -\alpha\log\pi_\theta(a'\mid s')\right],\quad a'\sim\pi_\theta,$$
$$J_\pi(\theta)=\mathbb E_{s\sim{\cal D},a\sim\pi_\theta}
  [\alpha\log\pi_\theta(a\mid s)-\min_iQ_{\phi_i}(s,a)].$$

These compact implementations share replay but leave every target visible.


In [ ]:
class DeterministicActor(nn.Module):
    def __init__(self, obs_dim, act_dim, limit):
        super().__init__(); self.net = mlp([obs_dim, 128, 128, act_dim], nn.Tanh); self.limit = limit
    def forward(self, state): return self.limit * self.net(state)


class ContinuousQ(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__(); self.net = mlp([obs_dim + act_dim, 128, 128, 1])
    def forward(self, state, action): return self.net(torch.cat([state, action], -1)).squeeze(-1)


def soft_update(target, source, tau):
    with torch.no_grad():
        for target_p, source_p in zip(target.parameters(), source.parameters()):
            target_p.mul_(1 - tau).add_(source_p, alpha=tau)


def train_ddpg(total_steps=2_000):
    torch.manual_seed(SEED)
    env = gym.make("Pendulum-v1")
    env.action_space.seed(SEED)
    obs_dim, act_dim = env.observation_space.shape[0], env.action_space.shape[0]
    limit = float(env.action_space.high[0])
    actor, critic = DeterministicActor(obs_dim, act_dim, limit).to(device), ContinuousQ(obs_dim, act_dim).to(device)
    actor_t, critic_t = DeterministicActor(obs_dim, act_dim, limit).to(device), ContinuousQ(obs_dim, act_dim).to(device)
    actor_t.load_state_dict(actor.state_dict()); critic_t.load_state_dict(critic.state_dict())
    actor_opt = torch.optim.Adam(actor.parameters(), 1e-3); critic_opt = torch.optim.Adam(critic.parameters(), 1e-3)
    replay, local_rng = Replay(), np.random.default_rng(SEED + 200)
    state, _ = env.reset(seed=SEED); ep_return, returns, q_losses, actor_losses = 0.0, [], [], []
    for step in range(total_steps):
        if step < 300:
            action = env.action_space.sample()
        else:
            with torch.no_grad(): action = actor(torch.as_tensor(state, dtype=torch.float32, device=device)).cpu().numpy()
            action = np.clip(action + local_rng.normal(0, .18 * limit, act_dim), -limit, limit)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        replay.add(state, action, reward, next_state, terminated); ep_return += reward; state = next_state
        if len(replay) >= 128:
            s, a, r, ns, d = replay.sample(128, local_rng)
            st = torch.as_tensor(s, dtype=torch.float32, device=device); at = torch.as_tensor(a, dtype=torch.float32, device=device)
            rt = torch.as_tensor(r, dtype=torch.float32, device=device); nst = torch.as_tensor(ns, dtype=torch.float32, device=device)
            dt = torch.as_tensor(d, dtype=torch.float32, device=device)
            with torch.no_grad(): y = rt + .99 * (1 - dt) * critic_t(nst, actor_t(nst))
            q_loss = F.mse_loss(critic(st, at), y)
            critic_opt.zero_grad(); q_loss.backward(); critic_opt.step()
            actor_loss = -critic(st, actor(st)).mean()
            actor_opt.zero_grad(); actor_loss.backward(); actor_opt.step()
            soft_update(actor_t, actor, .005); soft_update(critic_t, critic, .005)
            q_losses.append(float(q_loss))
            actor_losses.append(float(actor_loss))
        if done:
            returns.append(ep_return); state, _ = env.reset(); ep_return = 0.0
    env.close(); return np.asarray(returns), np.asarray(q_losses), np.asarray(actor_losses)


In [ ]:
LOG_STD_MIN, LOG_STD_MAX = -5.0, 2.0

class SquashedGaussianActor(nn.Module):
    def __init__(self, obs_dim, act_dim, limit):
        super().__init__(); self.body = mlp([obs_dim, 128, 128, 128]); self.mean = nn.Linear(128, act_dim)
        self.log_std = nn.Linear(128, act_dim); self.limit = limit

    def sample(self, state):
        h = self.body(state); mean = self.mean(h)
        log_std = self.log_std(h).clamp(LOG_STD_MIN, LOG_STD_MAX); std = log_std.exp()
        raw = Normal(mean, std).rsample(); squashed = torch.tanh(raw)
        action = self.limit * squashed
        logp = Normal(mean, std).log_prob(raw).sum(-1)
        logp -= torch.log(self.limit * (1 - squashed.pow(2)) + 1e-6).sum(-1)
        return action, logp


def train_sac(total_steps=2_000, entropy_coefficient=.2):
    torch.manual_seed(SEED)
    env = gym.make("Pendulum-v1")
    env.action_space.seed(SEED)
    obs_dim, act_dim = env.observation_space.shape[0], env.action_space.shape[0]
    limit = float(env.action_space.high[0])
    actor = SquashedGaussianActor(obs_dim, act_dim, limit).to(device)
    q1, q2 = ContinuousQ(obs_dim, act_dim).to(device), ContinuousQ(obs_dim, act_dim).to(device)
    q1_t, q2_t = ContinuousQ(obs_dim, act_dim).to(device), ContinuousQ(obs_dim, act_dim).to(device)
    q1_t.load_state_dict(q1.state_dict()); q2_t.load_state_dict(q2.state_dict())
    actor_opt = torch.optim.Adam(actor.parameters(), 3e-4)
    q_opt = torch.optim.Adam(list(q1.parameters()) + list(q2.parameters()), 3e-4)
    replay, local_rng = Replay(), np.random.default_rng(SEED + 300)
    state, _ = env.reset(seed=SEED); ep_return, returns, q_losses, actor_losses, entropies = 0.0, [], [], [], []
    for step in range(total_steps):
        if step < 300: action = env.action_space.sample()
        else:
            with torch.no_grad():
                action, _ = actor.sample(torch.as_tensor(state, dtype=torch.float32, device=device))
                action = action.cpu().numpy()
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        replay.add(state, action, reward, next_state, terminated); ep_return += reward; state = next_state
        if len(replay) >= 128:
            s, a, r, ns, d = replay.sample(128, local_rng)
            st = torch.as_tensor(s, dtype=torch.float32, device=device); at = torch.as_tensor(a, dtype=torch.float32, device=device)
            rt = torch.as_tensor(r, dtype=torch.float32, device=device); nst = torch.as_tensor(ns, dtype=torch.float32, device=device)
            dt = torch.as_tensor(d, dtype=torch.float32, device=device)
            with torch.no_grad():
                next_a, next_logp = actor.sample(nst)
                soft_value = torch.minimum(q1_t(nst, next_a), q2_t(nst, next_a)) - entropy_coefficient * next_logp
                y = rt + .99 * (1 - dt) * soft_value
            q_loss = F.mse_loss(q1(st, at), y) + F.mse_loss(q2(st, at), y)
            q_opt.zero_grad(); q_loss.backward(); q_opt.step()
            sampled_a, logp = actor.sample(st)
            actor_loss = (entropy_coefficient * logp - torch.minimum(q1(st, sampled_a), q2(st, sampled_a))).mean()
            actor_opt.zero_grad(); actor_loss.backward(); actor_opt.step()
            soft_update(q1_t, q1, .005); soft_update(q2_t, q2, .005)
            q_losses.append(float(q_loss))
            actor_losses.append(float(actor_loss))
            entropies.append(float(-logp.mean()))
        if done:
            returns.append(ep_return); state, _ = env.reset(); ep_return = 0.0
    env.close()
    return (
        np.asarray(returns),
        np.asarray(q_losses),
        np.asarray(actor_losses),
        np.asarray(entropies),
    )


continuous_steps = 1_200 if QUICK else 40_000
ddpg_returns, ddpg_losses, ddpg_actor_losses = train_ddpg(continuous_steps)
sac_returns, sac_losses, sac_actor_losses, sac_entropies = train_sac(continuous_steps)
fig, ax = plt.subplots(1, 3, figsize=(14, 3.3))
ax[0].plot(ddpg_returns, label="DDPG"); ax[0].plot(sac_returns, label="SAC")
ax[0].set(title="Pendulum return", xlabel="episode"); ax[0].legend()
ax[1].plot(ddpg_losses, alpha=.7, label="DDPG critic")
ax[1].plot(sac_losses, alpha=.7, label="SAC twin critics")
ax[1].set(title="Critic loss", xlabel="gradient step", yscale="log"); ax[1].legend()
ax[2].plot(ddpg_actor_losses, alpha=.7, label="DDPG actor")
ax[2].plot(sac_actor_losses, alpha=.7, label="SAC actor")
ax[2].plot(sac_entropies, alpha=.6, label="SAC entropy")
ax[2].set(title="Actor objectives / entropy", xlabel="gradient step"); ax[2].legend()
plt.tight_layout()


## 8. One view of the algorithm family

Every method above decides (i) which distribution supplies data, (ii) which
operator defines a target, and (iii) how that target is projected into a
representable function class.

- DP knows $P,R$ and applies the operator exactly.
- Monte Carlo samples an unbiased full return with potentially high variance.
- TD samples a reward/transition and bootstraps, trading variance for bias and
  target coupling.
- DQN stabilizes approximate off-policy optimality updates with replay and a
  delayed target.
- REINFORCE differentiates the trajectory distribution; actor--critic replaces
  its return with a learned control variate/advantage.
- DDPG differentiates through a deterministic critic; SAC regularizes the
  control problem with entropy and retains a stochastic actor.

| algorithm | failure modes worth diagnosing |
|---|---|
| policy/value iteration | model error, state explosion, improper undiscounted policies |
| Monte Carlo | long-horizon variance, importance-weight tails, insufficient exploring starts |
| SARSA / Expected SARSA | persistent behavior-policy bias, sensitivity to step-size and exploration decay |
| Q-learning / Double Q | max bias, rare-action undercoverage, deadly-triad divergence after approximation |
| DQN | replay support mismatch, target drift, exploding Q scale, correlated evaluation seeds |
| REINFORCE | extreme gradient variance, weak baseline, premature entropy collapse |
| advantage actor--critic | critic bias leaking into the actor, bootstrapping at truncations, loss-scale imbalance |
| DDPG | brittle external action noise, critic extrapolation, actor saturation |
| SAC | entropy-temperature mismatch, log-probability correction errors, critic underestimation |

**Diagnostics to retain.** Learning curves alone conceal support mismatch,
extrapolation, and unstable targets. Record visitation, policy entropy, target
and TD-error distributions, gradient norms, Q scale, Bellman residuals, and
seed-level outcome distributions. The remaining notebooks make those objects
first-class experimental data.
